In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into DataFrames (kept in kernel variables for later reuse)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# timezone for display (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

def summarize_df(df, filename, name_field=None, cmdb_field='cmdb_id',
                 max_cmdb=20, max_names=50):
    # Ensure timestamp handling (timestamps are seconds)
    if 'timestamp' in df.columns and not df['timestamp'].empty:
        ts = pd.to_datetime(df['timestamp'], unit='s', utc=True)
        ts_local = ts.dt.tz_convert(tz)
        earliest = ts_local.min()
        latest = ts_local.max()
        earliest_str = earliest.isoformat()
        latest_str = latest.isoformat()
    else:
        earliest_str = None
        latest_str = None

    # row count
    row_count = len(df)

    # unique cmdb ids (limit)
    if cmdb_field in df.columns:
        unique_cmdb = list(pd.Index(df[cmdb_field].dropna().unique())[:max_cmdb])
    else:
        unique_cmdb = []

    # unique KPI/trace/log field names (limit)
    if name_field and name_field in df.columns:
        unique_names = list(pd.Index(df[name_field].dropna().unique())[:max_names])
    else:
        # fallback: list other meaningful string columns (e.g., 'message' for error_logs)
        other_cols = [c for c in df.columns if c not in ('timestamp', cmdb_field)]
        # take unique values from first suitable column up to limit
        unique_names = []
        if other_cols:
            col = other_cols[0]
            unique_names = list(pd.Index(df[col].dropna().unique())[:max_names])

    return {
        'filename': filename,
        'row_count': row_count,
        'earliest_timestamp': earliest_str,
        'latest_timestamp': latest_str,
        'unique_cmdb_ids': unique_cmdb,
        'unique_names': unique_names
    }

# Build summaries for each file using the appropriate name_field
summary_metric = summarize_df(metric_df, "metric.csv", name_field='kpi_name')
summary_trace = summarize_df(trace_df, "trace.csv", name_field='trace_name')
summary_log = summarize_df(log_df, "log.csv", name_field='log_name')
# For error_logs, use 'message' as the representative field
summary_error_logs = summarize_df(error_logs_df, "error_logs.csv", name_field='message')

# Create a compact summary DataFrame (one row per file)
summary_df = pd.DataFrame([summary_metric, summary_trace, summary_log, summary_error_logs])

# Keep final output compact: show only the requested columns and ensure lists are truncated
summary_df[['filename', 'row_count', 'earliest_timestamp', 'latest_timestamp',
            'unique_cmdb_ids', 'unique_names']]

```
Out[1]:
```
```python
# Summarize the previously computed summaries in plain English as a single compact string.
# Reuse summary_df from the kernel state.
rows = []
for _, r in summary_df.iterrows():
    fn = r['filename']
    rc = int(r['row_count'])
    start = r['earliest_timestamp'] if pd.notna(r['earliest_timestamp']) else None
    end = r['latest_timestamp'] if pd.notna(r['latest_timestamp']) else None
    cmdbs = r['unique_cmdb_ids'] if isinstance(r['unique_cmdb_ids'], list) else []
    names = r['unique_names'] if isinstance(r['unique_names'], list) else []
    cmdb_preview = ', '.join(cmdbs[:20]) if cmdbs else '[]'
    names_preview = ', '.join(names[:50]) if names else '[]'
    rows.append(f"- {fn}: {rc} rows; time range {start} to {end}; cmdb_ids: [{cmdb_preview}]; names/sample: [{names_preview}]")

summary_text = "Summary of loaded telemetry files:\n" + "\n".join(rows)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

filename  row_count         earliest_timestamp           latest_timestamp                                    unique_cmdb_ids                                       unique_names
0      metric.csv      22148  2021-03-23T15:00:00+08:00  2021-03-23T15:30:00+08:00  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...  [JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM...
1       trace.csv       7203  2021-03-23T15:00:00+08:00  2021-03-23T15:30:00+08:00  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...  [trace.from_root.duration_mean, trace.from_roo...
2         log.csv        473  2021-03-23T15:00:00+08:00  2021-03-23T15:30:00+08:00  [Tomcat01, Tomcat02, Tomcat03, Tomcat04, apach...  [log.error_count, log.gc_pause_max, log.row_co...
3  error_logs.csv          0                       None                       None                                                 []                                                 []```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing loaded DataFrames if present, otherwise load from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

# timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# 1) Aggregate full metric_df by (cmdb_id, kpi_name) to compute global statistics
# Ensure 'value' is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Define aggregation using named aggregation
agg_funcs = {
    'count': ('value', 'count'),
    'min': ('value', 'min'),
    'p50': ('value', lambda x: x.quantile(0.50)),
    'p90': ('value', lambda x: x.quantile(0.90)),
    'p95': ('value', lambda x: x.quantile(0.95)),
    'p99': ('value', lambda x: x.quantile(0.99)),
    'max': ('value', 'max'),
    'mean': ('value', 'mean'),
    'std': ('value', 'std')
}

thresholds_df = (metric_df
                 .groupby(['cmdb_id', 'kpi_name'], dropna=False)
                 .agg(**agg_funcs)
                 .reset_index()
                )

# Keep thresholds_df in kernel (do not print full table)

# 2) From thresholds_df, return top 20 rows sorted by p95 descending with selected columns
top20_df = (thresholds_df
            .sort_values('p95', ascending=False)
            .loc[:, ['cmdb_id', 'kpi_name', 'count', 'p95', 'p99', 'mean', 'std']]
            .head(20)
           )

# 3) Filter metric.csv, trace.csv, log.csv to incident time range 2021-03-23 15:00 to 15:30 (UTC+8) inclusive
start_local = tz.localize(pd.Timestamp("2021-03-23 15:00:00"))
end_local = tz.localize(pd.Timestamp("2021-03-23 15:30:00"))

def filter_by_local_range(df):
    # convert timestamp to aware datetime in UTC then to local tz
    ts_local = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    mask = (ts_local >= start_local) & (ts_local <= end_local)
    filtered = df.loc[mask.values].copy()
    # attach a local timestamp column for convenience (not required to display full data)
    filtered['_timestamp_local'] = ts_local.loc[mask].values
    return filtered

metric_incident_df = filter_by_local_range(metric_df)
trace_incident_df = filter_by_local_range(trace_df)
log_incident_df = filter_by_local_range(log_df)

# 4) Compact summary of the filtered datasets
def compact_summary(filtered_df, filename, cmdb_field='cmdb_id'):
    row_count = int(filtered_df.shape[0])
    if row_count > 0:
        earliest = filtered_df['_timestamp_local'].min().isoformat()
        latest = filtered_df['_timestamp_local'].max().isoformat()
        unique_cmdbs = list(pd.Index(filtered_df[cmdb_field].dropna().unique())[:20])
    else:
        earliest = None
        latest = None
        unique_cmdbs = []
    return {
        'filename': filename,
        'filtered_row_count': row_count,
        'earliest_timestamp_in_filtered': earliest,
        'latest_timestamp_in_filtered': latest,
        'unique_cmdb_ids': unique_cmdbs
    }

summary_metric_incident = compact_summary(metric_incident_df, 'metric.csv')
summary_trace_incident = compact_summary(trace_incident_df, 'trace.csv')
summary_log_incident = compact_summary(log_incident_df, 'log.csv')

summary_filtered_df = pd.DataFrame([summary_metric_incident,
                                    summary_trace_incident,
                                    summary_log_incident])

# Final compact outputs: top20_df and summary_filtered_df
top20_df, summary_filtered_df

```
Out[2]:
```
```python
# Summarize results in plain English using existing kernel variables (top20_df, summary_filtered_df, thresholds_df)
# Display a compact summary string.

# Prepare top 5 highlights from top20_df
highlights = []
for _, r in top20_df.head(5).iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    count = int(r['count'])
    p95 = float(r['p95'])
    p99 = float(r['p99'])
    mean = float(r['mean']) if pd.notna(r['mean']) else float('nan')
    std = float(r['std']) if pd.notna(r['std']) else float('nan')
    highlights.append(f"{cmdb} / {kpi}: p95={p95:.3e}, p99={p99:.3e}, count={count}, mean={mean:.3e}, std={std:.3e}")

# Summaries of filtered datasets
filtered_lines = []
for _, r in summary_filtered_df.iterrows():
    fn = r['filename']
    rc = int(r['filtered_row_count'])
    start = r['earliest_timestamp_in_filtered']
    end = r['latest_timestamp_in_filtered']
    cmdbs = ', '.join(r['unique_cmdb_ids'][:8]) if r['unique_cmdb_ids'] else '[]'
    filtered_lines.append(f"{fn}: {rc} rows; time range {start} to {end}; sample cmdbs: [{cmdbs}]")

summary_text = (
    "Compact summary:\n"
    "- thresholds_df has been computed and stored (per-(cmdb_id,kpi_name) global stats).\n"
    "- Top 20 by p95 (top 5 highlights shown):\n  " + "\n  ".join(highlights) + "\n"
    "- Observations: the top p95 entries are dominated by Mysql metrics (very large numeric values, e+11–e+12 range) "
    "with counts mostly 29, indicating heavy MySQL I/O/rows metrics. Several Tomcat JVM memory KPIs appear with "
    "p95 ~4.187619e9 and std = 0 (constant samples, count=6).\n"
    "- Filtered datasets for the incident window saved as metric_incident_df, trace_incident_df, log_incident_df.\n"
    "- Filtered dataset summaries:\n  " + "\n  ".join(filtered_lines)
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count           p95           p99          mean           std
441    Mysql02               Mysql-MySQL_3306_Innodb data written     29  2.756001e+12  2.756055e+12  1.330275e+12  1.401337e+12
430    Mysql02  Mysql-MySQL_3306_Innodb buffer pool read requests     29  1.665184e+12  1.665184e+12  8.038784e+11  8.468217e+11
451    Mysql02             Mysql-MySQL_3306_Innodb os log written     29  7.194566e+11  7.194936e+11  3.472389e+11  3.657882e+11
304    Mysql01               Mysql-MySQL_3306_Innodb data written     29  6.207077e+11  6.207287e+11  3.209866e+11  3.155913e+11
481    Mysql02                         Mysql-MySQL_3306_Rows Read     29  5.444171e+11  5.444171e+11  2.628212e+11  2.768611e+11
412    Mysql02                 Mysql-MySQL_3306_Handler Read Prev     29  5.442331e+11  5.442332e+11  2.627324e+11  2.767676e+11
389    Mysql02                        Mysql-MySQL_3306_Bytes Sent     29  1.836017e+11  1.836112e+11  8.860469e+10  9.333785e+10
252    Mysql01                        Mysql-MySQL_3306_Bytes Sent     29  7.734797e+10  7.735106e+10  3.999570e+10  3.932343e+10
438    Mysql02                  Mysql-MySQL_3306_Innodb data read     29  3.317277e+10  3.317277e+10  1.601444e+10  1.686993e+10
314    Mysql01             Mysql-MySQL_3306_Innodb os log written     29  9.082599e+09  9.082912e+09  4.696670e+09  4.617725e+09
871   Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
1115  Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
750   Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
869   Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
993   Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
752   Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
1113  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
991   Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  0.000000e+00
277    Mysql01             Mysql-MySQL_3306_Handler Read Rnd Next     29  3.576930e+09  3.577080e+09  1.849575e+09  1.818486e+09
344    Mysql01                         Mysql-MySQL_3306_Rows Read     29  3.231124e+09  3.231276e+09  1.670701e+09  1.642618e+09,      filename  filtered_row_count earliest_timestamp_in_filtered latest_timestamp_in_filtered                                    unique_cmdb_ids
0  metric.csv               22148            2021-03-23T07:00:00          2021-03-23T07:30:00  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...
1   trace.csv                7203            2021-03-23T07:00:00          2021-03-23T07:30:00  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...
2     log.csv                 473            2021-03-23T07:00:00          2021-03-23T07:30:00  [Tomcat01, Tomcat02, Tomcat03, Tomcat04, apach...)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse thresholds_df and metric_incident_df from kernel state
# Ensure they exist
try:
    thresholds_df
    metric_incident_df
except NameError as e:
    raise

# Merge to bring p95 threshold into each incident metric row
merged = metric_incident_df.merge(
    thresholds_df[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left',
    validate='m:1'
)

# Flag anomalies where value >= p95 (and p95 is not null)
anomalies_df = merged.loc[merged['p95'].notna() & (merged['value'].astype(float) >= merged['p95'].astype(float))].copy()

# Save anomalies_df in kernel (full rows retained but not printed)
# anomalies_df is now available in kernel

# If no anomalies found with p95, we will not fallback to p90 in this run (per instructions)
if anomalies_df.empty:
    # Create empty anomalies_agg_df with expected columns to keep kernel variables consistent
    anomalies_agg_df = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'anomaly_count',
        'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
        'max_value', 'p95_threshold', 'severity_ratio'
    ])
    top20_anomalies = anomalies_agg_df.head(20)
    top_components_df = pd.DataFrame(columns=[
        'cmdb_id', 'total_anomalies', 'distinct_kpis_with_anomalies', 'earliest_anomaly_timestamp_overall'
    ]).head(10)
else:
    # Aggregate anomalies by (cmdb_id, kpi_name)
    def _min_iso(x):
        return x.min().isoformat()
    def _max_iso(x):
        return x.max().isoformat()

    anomalies_agg_df = (
        anomalies_df
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('_timestamp_local', lambda x: x.min().isoformat()),
            latest_anomaly_timestamp=('_timestamp_local', lambda x: x.max().isoformat()),
            max_value=('value', 'max'),
            p95_threshold=('p95', 'first')
        )
        .reset_index()
    )

    # compute severity_ratio safely
    anomalies_agg_df['severity_ratio'] = anomalies_agg_df.apply(
        lambda r: (np.nan if pd.isna(r['p95_threshold']) or r['p95_threshold'] == 0 else float(r['max_value']) / float(r['p95_threshold'])),
        axis=1
    )

    # Top 20 by anomaly_count desc
    top20_anomalies = (
        anomalies_agg_df
        .sort_values('anomaly_count', ascending=False)
        .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value', 'p95_threshold', 'severity_ratio']]
        .head(20)
        .reset_index(drop=True)
    )

    # Top 10 components by total anomaly_count across KPIs
    top_components_df = (
        anomalies_df
        .groupby('cmdb_id', dropna=False)
        .agg(
            total_anomalies=('value', 'size'),
            distinct_kpis_with_anomalies=('kpi_name', 'nunique'),
            earliest_anomaly_timestamp_overall=('_timestamp_local', lambda x: x.min().isoformat())
        )
        .reset_index()
        .sort_values('total_anomalies', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

# Return the two compact outputs only
top20_anomalies, top_components_df

```
Out[3]:
```
```python
# Build a concise plain-English summary using existing kernel variables.
# The final output is the string variable `summary_text`.

import pandas as pd
import math

# Reuse kernel variables: anomalies_df, anomalies_agg_df, top20_anomalies, top_components_df
try:
    anomalies_df
    anomalies_agg_df
    top20_anomalies
    top_components_df
except NameError:
    summary_text = "Required kernel variables are missing."
else:
    if anomalies_df.empty:
        summary_text = "No metric anomalies detected using the global p95 thresholds in the incident window."
    else:
        total_anomalies = int(anomalies_df.shape[0])
        distinct_anomaly_pairs = int(anomalies_agg_df.shape[0])
        earliest_overall = anomalies_df['_timestamp_local'].min().isoformat()
        latest_overall = anomalies_df['_timestamp_local'].max().isoformat()

        # Top 5 anomalous KPI entries (from top20_anomalies)
        top_kpis_lines = []
        for _, r in top20_anomalies.head(5).iterrows():
            cmdb = r['cmdb_id']
            kpi = r['kpi_name']
            cnt = int(r['anomaly_count'])
            mv = float(r['max_value']) if not pd.isna(r['max_value']) else float('nan')
            thr = float(r['p95_threshold']) if not pd.isna(r['p95_threshold']) else float('nan')
            sev = float(r['severity_ratio']) if not pd.isna(r['severity_ratio']) else float('nan')
            mv_s = f"{mv:.3e}" if not math.isnan(mv) else "NaN"
            thr_s = f"{thr:.3e}" if not math.isnan(thr) else "NaN"
            sev_s = f"{sev:.3f}" if not math.isnan(sev) else "NaN"
            top_kpis_lines.append(f"{cmdb} / {kpi}: anomalies={cnt}, max={mv_s}, p95={thr_s}, severity_ratio={sev_s}")

        # Top 5 components by total anomalies
        top_components_lines = []
        for _, r in top_components_df.head(5).iterrows():
            cmdb = r['cmdb_id']
            tot = int(r['total_anomalies'])
            distinct_kpis = int(r['distinct_kpis_with_anomalies'])
            earliest = r['earliest_anomaly_timestamp_overall']
            top_components_lines.append(f"{cmdb}: total_anomalies={tot}, distinct_kpis={distinct_kpis}, first_seen={earliest}")

        summary_text = (
            "Metric anomaly detection (using global p95 thresholds) — incident window:\n"
            f"- Total anomaly datapoints flagged: {total_anomalies}\n"
            f"- Distinct (cmdb_id, kpi_name) anomaly groups: {distinct_anomaly_pairs}\n"
            f"- Time span of flagged anomalies: {earliest_overall} to {latest_overall}\n\n"
            "- Top anomalous KPIs (sample top 5):\n  " + "\n  ".join(top_kpis_lines) + "\n\n"
            "- Top components by anomaly count (top 5):\n  " + "\n  ".join(top_components_lines) + "\n\n"
            "Notes:\n"
            "- MySQL instances (Mysql01, Mysql02) have the highest total anomaly counts.\n"
            "- Many Tomcat JVM and request/session KPIs show repeated anomalies (counts=31), some with severity_ratio ~1.0.\n"
            "- Some KPIs have p95_threshold == 0, producing undefined severity_ratio (NaN).\n"
            "- All results are saved in kernel variables: anomalies_df (flagged rows), anomalies_agg_df (aggregated groups).\n"
        )

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp     max_value  p95_threshold  severity_ratio
0   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
1       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31        2021-03-23T07:00:00  5.865866e+07   5.865866e+07             1.0
2   Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
3       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31        2021-03-23T07:00:00  2.040136e+09   2.040136e+09             1.0
4   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31        2021-03-23T07:00:00  2.304870e+05   2.304870e+05             1.0
5   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31        2021-03-23T07:00:00  1.390000e+02   1.390000e+02             1.0
6   Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
7   Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
8   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
9   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31        2021-03-23T07:00:00  5.000000e+02   5.000000e+02             1.0
10  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31        2021-03-23T07:00:00  1.000000e+02   1.000000e+02             1.0
11  Tomcat02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31        2021-03-23T07:00:00  3.000000e+00   3.000000e+00             1.0
12  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
13  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
14  Tomcat02        Tomcat-Sessions_7441--_SESSIONActiveCounter             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
15  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31        2021-03-23T07:00:00  5.000000e+02   5.000000e+02             1.0
16  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
17  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
18      MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
19  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN,     cmdb_id  total_anomalies  distinct_kpis_with_anomalies earliest_anomaly_timestamp_overall
0   Mysql01             1958                           137                2021-03-23T07:00:00
1   Mysql02             1793                           137                2021-03-23T07:00:00
2  Tomcat03             1247                           122                2021-03-23T07:00:00
3  Tomcat04             1219                           122                2021-03-23T07:00:00
4  Tomcat01             1210                           120                2021-03-23T07:00:00
5  Tomcat02             1129                           122                2021-03-23T07:00:00
6      MG01              880                           104                2021-03-23T07:00:00
7      MG02              709                           105                2021-03-23T07:00:00
8   Redis02              605                            76                2021-03-23T07:00:00
9      IG01              141                            18                2021-03-23T07:00:00)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables if present
try:
    trace_df
    trace_incident_df
except NameError as e:
    raise

tz = pytz.timezone('Asia/Shanghai')

# Ensure value is numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_incident_df['value'] = pd.to_numeric(trace_incident_df['value'], errors='coerce')

# 1) Compute global thresholds on full trace_df grouped by (cmdb_id, trace_name)
agg_funcs = {
    'count': ('value', 'count'),
    'min': ('value', 'min'),
    'p50': ('value', lambda x: x.quantile(0.50)),
    'p90': ('value', lambda x: x.quantile(0.90)),
    'p95': ('value', lambda x: x.quantile(0.95)),
    'p99': ('value', lambda x: x.quantile(0.99)),
    'max': ('value', 'max'),
    'mean': ('value', 'mean'),
    'std': ('value', 'std')
}

trace_thresholds_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(**agg_funcs)
    .reset_index()
)

# 2) Join thresholds with incident-window traces and flag anomalies using p95 first
# Merge p95 and p90 into incident rows for fallback if needed
merged = trace_incident_df.merge(
    trace_thresholds_df[['cmdb_id', 'trace_name', 'p95', 'p90']],
    on=['cmdb_id', 'trace_name'],
    how='left',
    validate='m:1'
)

# Flag using p95
merged['threshold_used_value'] = merged['p95']
merged['threshold_used_name'] = 'p95'
trace_anomalies_df = merged.loc[merged['threshold_used_value'].notna() & (merged['value'] >= merged['threshold_used_value'])].copy()

# If no anomalies found using p95, fallback to p90 (and report used threshold)
threshold_used = 'p95'
if trace_anomalies_df.empty:
    merged['threshold_used_value'] = merged['p90']
    merged['threshold_used_name'] = 'p90'
    trace_anomalies_df = merged.loc[merged['threshold_used_value'].notna() & (merged['value'] >= merged['threshold_used_value'])].copy()
    threshold_used = 'p90'

# Save trace_anomalies_df in kernel
# 3) Aggregate trace_anomalies_df by (cmdb_id, trace_name)
if trace_anomalies_df.empty:
    trace_anomalies_agg_df = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'anomaly_count',
        'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
        'max_value', 'p95_threshold', 'severity_ratio'
    ])
else:
    # Ensure local timestamp column exists in trace_anomalies_df (should be _timestamp_local from earlier filtering)
    if '_timestamp_local' not in trace_anomalies_df.columns:
        ts_local = pd.to_datetime(trace_anomalies_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        trace_anomalies_df['_timestamp_local'] = ts_local.values

    def _min_iso(x): return x.min().isoformat()
    def _max_iso(x): return x.max().isoformat()

    # Use the threshold value that was actually used (p95 or p90) as p95_threshold in output for consistency,
    # and record severity_ratio = max_value / threshold_used_value
    trace_anomalies_agg_df = (
        trace_anomalies_df
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('_timestamp_local', lambda x: x.min().isoformat()),
            latest_anomaly_timestamp=('_timestamp_local', lambda x: x.max().isoformat()),
            max_value=('value', 'max'),
            p95_threshold=('threshold_used_value', 'first')
        )
        .reset_index()
    )

    trace_anomalies_agg_df['severity_ratio'] = trace_anomalies_agg_df.apply(
        lambda r: (np.nan if pd.isna(r['p95_threshold']) or r['p95_threshold'] == 0 else float(r['max_value']) / float(r['p95_threshold'])),
        axis=1
    )

# 4a) Top 20 trace anomaly groups by anomaly_count desc
top20_trace_anomalies = (
    trace_anomalies_agg_df
    .sort_values('anomaly_count', ascending=False)
    .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value', 'p95_threshold', 'severity_ratio']]
    .head(20)
    .reset_index(drop=True)
)

# 4b) Top 10 components by total trace anomaly count
if trace_anomalies_df.empty:
    top_trace_components_df = pd.DataFrame(columns=[
        'cmdb_id', 'total_trace_anomalies', 'distinct_trace_names_with_anomalies', 'earliest_anomaly_timestamp_overall'
    ])
else:
    top_trace_components_df = (
        trace_anomalies_df
        .groupby('cmdb_id', dropna=False)
        .agg(
            total_trace_anomalies=('value', 'size'),
            distinct_trace_names_with_anomalies=('trace_name', 'nunique'),
            earliest_anomaly_timestamp_overall=('_timestamp_local', lambda x: x.min().isoformat())
        )
        .reset_index()
        .sort_values('total_trace_anomalies', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

# Expose which threshold was used as a small scalar for the caller's info
threshold_used

# Final compact outputs
top20_trace_anomalies, top_trace_components_df

```
Out[4]:
```
```python
# Summarize trace anomaly detection results in plain English (compact).
# Reuse kernel variables: trace_thresholds_df, trace_anomalies_df, trace_anomalies_agg_df, top20_trace_anomalies, top_trace_components_df

try:
    trace_anomalies_df
    trace_anomalies_agg_df
    top20_trace_anomalies
    top_trace_components_df
except NameError:
    summary_text = "Trace anomaly variables not found in kernel."
else:
    import math
    total_trace_anomalies = int(trace_anomalies_df.shape[0])
    distinct_trace_groups = int(trace_anomalies_agg_df.shape[0])
    earliest = (trace_anomalies_df['_timestamp_local'].min().isoformat() if '_timestamp_local' in trace_anomalies_df.columns and not trace_anomalies_df.empty else None)
    latest = (trace_anomalies_df['_timestamp_local'].max().isoformat() if '_timestamp_local' in trace_anomalies_df.columns and not trace_anomalies_df.empty else None)
    threshold_names = sorted(set(trace_anomalies_df['threshold_used_name'].dropna().unique())) if 'threshold_used_name' in trace_anomalies_df.columns else []

    # Top 5 anomalous trace groups (sample)
    top_kpis_lines = []
    for _, r in top20_trace_anomalies.head(5).iterrows():
        cmdb = r['cmdb_id']
        name = r['trace_name']
        cnt = int(r['anomaly_count'])
        mv = float(r['max_value']) if not pd.isna(r['max_value']) else float('nan')
        thr = float(r['p95_threshold']) if not pd.isna(r['p95_threshold']) else float('nan')
        sev = float(r['severity_ratio']) if not pd.isna(r['severity_ratio']) else float('nan')
        mv_s = f"{mv:.3f}" if not math.isnan(mv) else "NaN"
        thr_s = f"{thr:.3f}" if not math.isnan(thr) else "NaN"
        sev_s = f"{sev:.3f}" if not math.isnan(sev) else "NaN"
        top_kpis_lines.append(f"{cmdb} / {name}: anomalies={cnt}, max={mv_s}, thr={thr_s}, severity={sev_s}")

    # Top 5 components by trace anomaly count
    top_components_lines = []
    for _, r in top_trace_components_df.head(5).iterrows():
        cmdb = r['cmdb_id']
        tot = int(r['total_trace_anomalies'])
        distinct = int(r['distinct_trace_names_with_anomalies'])
        first = r['earliest_anomaly_timestamp_overall']
        top_components_lines.append(f"{cmdb}: total={tot}, distinct_traces={distinct}, first_seen={first}")

    summary_text = (
        "Trace anomaly detection summary (incident window):\n"
        f"- Global trace thresholds computed and stored in trace_thresholds_df.\n"
        f"- Threshold used for flagging: {', '.join(threshold_names) if threshold_names else 'unknown'} (p95 preferred, p90 fallback if used).\n"
        f"- Total trace datapoints flagged: {total_trace_anomalies}\n"
        f"- Distinct (cmdb_id, trace_name) anomaly groups: {distinct_trace_groups}\n"
        f"- Time span of flagged trace anomalies: {earliest} to {latest}\n\n"
        "- Sample top anomalous trace groups (top 5):\n  " + "\n  ".join(top_kpis_lines) + "\n\n"
        "- Top components by trace anomaly count (top 5):\n  " + "\n  ".join(top_components_lines) + "\n\n"
        "Notes:\n"
        "- Tomcat nodes (Tomcat01/02/03/04) and MG nodes have many trace anomalies (high counts).\n"
        "- Some trace metrics show large severity ratios (max / threshold > 2), indicating notable deviations.\n"
        "- Full results saved in kernel variables: trace_thresholds_df, trace_anomalies_df, trace_anomalies_agg_df, top20_trace_anomalies, top_trace_components_df."
    )

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                     trace_name  anomaly_count earliest_anomaly_timestamp    max_value  p95_threshold  severity_ratio
0   Tomcat01        trace.self.duration_p95             17        2021-03-23T07:01:00     1.705450       1.004000        1.698655
1   Tomcat03        trace.self.duration_p95             16        2021-03-23T07:01:00     1.500900       1.004000        1.494920
2   Tomcat04        trace.self.duration_p95             14        2021-03-23T07:00:00     2.507800       1.005000        2.495323
3   Tomcat02        trace.self.duration_p95             10        2021-03-23T07:03:00     1.005000       1.005000        1.000000
4   dockerB1        trace.self.duration_p95              8        2021-03-23T07:00:00     1.006000       1.006000        1.000000
5   dockerA2           trace.self.row_count              2        2021-03-23T07:20:00  4838.000000    3283.500000        1.473428
6   dockerA2    trace.to_MG01.duration_mean              2        2021-03-23T07:27:00     1.027000       0.700105        1.466924
7   dockerA2     trace.to_MG01.duration_p95              2        2021-03-23T07:26:00     4.393800       2.332375        1.883831
8       IG01       trace.self.duration_mean              2        2021-03-23T07:27:00     2.361762       1.074240        2.198542
9   dockerA2    trace.to_MG02.duration_mean              2        2021-03-23T07:27:00     1.072000       0.611897        1.751927
10  dockerA2     trace.to_MG02.duration_p95              2        2021-03-23T07:26:00     4.775650       1.728900        2.762248
11  dockerA2        trace.to_MG02.row_count              2        2021-03-23T07:21:00    98.000000      85.000000        1.152941
12  dockerB1  trace.from_MG01.duration_mean              2        2021-03-23T07:21:00     5.783667       1.088315        5.314334
13  dockerA2  trace.from_MG02.duration_mean              2        2021-03-23T07:29:00     0.965625       0.376787        2.562787
14  dockerA2   trace.from_MG02.duration_p95              2        2021-03-23T07:29:00     2.111100       1.521950        1.387102
15  dockerA2      trace.from_MG02.row_count              2        2021-03-23T07:28:00   367.000000     230.500000        1.592191
16  dockerA2        trace.to_MG01.row_count              2        2021-03-23T07:20:00    93.000000      81.500000        1.141104
17  dockerA2        trace.self.duration_p95              2        2021-03-23T07:29:00     1.007200       0.309000        3.259547
18  dockerA1     trace.to_MG01.duration_p95              2        2021-03-23T07:26:00     5.936700       2.516477        2.359131
19  dockerA1        trace.to_MG01.row_count              2        2021-03-23T07:21:00   107.000000      87.850000        1.217985,     cmdb_id  total_trace_anomalies  distinct_trace_names_with_anomalies earliest_anomaly_timestamp_overall
0      MG02                     81                                   42                2021-03-23T07:00:00
1      MG01                     81                                   42                2021-03-23T07:00:00
2  Tomcat01                     48                                   18                2021-03-23T07:00:00
3  Tomcat03                     47                                   18                2021-03-23T07:00:00
4  Tomcat04                     45                                   18                2021-03-23T07:00:00
5  Tomcat02                     41                                   18                2021-03-23T07:00:00
6  dockerB1                     39                                   18                2021-03-23T07:00:00
7      IG02                     33                                   18                2021-03-23T07:00:00
8      IG01                     33                                   18                2021-03-23T07:00:00
9  dockerA1                     33                                   18                2021-03-23T07:00:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables
try:
    anomalies_df
    anomalies_agg_df
    metric_incident_df
    thresholds_df
except NameError as e:
    raise

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp fields exist and are numeric
# anomalies_df should contain 'timestamp' (epoch seconds) and '_timestamp_local' (aware tz) from prior steps.
if '_timestamp_local' not in anomalies_df.columns:
    anomalies_df['_timestamp_local'] = pd.to_datetime(anomalies_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Work on a copy to avoid mutating original unexpectedly
anom = anomalies_df.copy()

# Convert timestamp to integer seconds (should be minute-granular)
anom['ts_sec'] = anom['timestamp'].astype(int)

# Sort for grouping
anom = anom.sort_values(['cmdb_id', 'kpi_name', 'ts_sec']).reset_index(drop=True)

# For each (cmdb_id, kpi_name), identify consecutive runs where difference == 60s
# Compute gap indicator where gap > 60 (break)
anom['prev_ts'] = anom.groupby(['cmdb_id', 'kpi_name'])['ts_sec'].shift(1)
anom['gap'] = (anom['ts_sec'] - anom['prev_ts']).fillna(9999999).astype(int)
# Start of a new fault interval when gap != 60 (i.e., gap>60 or NaN)
anom['new_interval'] = (anom['gap'] != 60).astype(int)
# Cumulative sum to assign interval ids within each group
anom['interval_id'] = anom.groupby(['cmdb_id', 'kpi_name'])['new_interval'].cumsum()

# Build faults_df by grouping by cmdb_id, kpi_name, interval_id
grouped = anom.groupby(['cmdb_id', 'kpi_name', 'interval_id'], dropna=False)

faults_list = []
for (cmdb, kpi, interval_id), g in grouped:
    start_ts_sec = int(g['ts_sec'].min())
    end_ts_sec = int(g['ts_sec'].max())
    # Convert to local ISO strings
    start_iso = pd.to_datetime(start_ts_sec, unit='s', utc=True).tz_convert(tz).isoformat()
    end_iso = pd.to_datetime(end_ts_sec, unit='s', utc=True).tz_convert(tz).isoformat()
    length_minutes = int(g.shape[0])  # consecutive minutes count
    max_value = float(g['value'].max())
    # lookup p95 threshold from thresholds_df
    thr_row = thresholds_df[
        (thresholds_df['cmdb_id'] == cmdb) & (thresholds_df['kpi_name'] == kpi)
    ]
    p95_threshold = float(thr_row['p95'].iloc[0]) if not thr_row.empty and pd.notna(thr_row['p95'].iloc[0]) else np.nan
    severity_ratio = (np.nan if pd.isna(p95_threshold) or p95_threshold == 0 else max_value / p95_threshold)
    faults_list.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'interval_id': int(interval_id),
        'start_timestamp': start_iso,
        'end_timestamp': end_iso,
        'length_in_minutes': length_minutes,
        'max_value_in_interval': max_value,
        'p95_threshold': p95_threshold,
        'severity_ratio': severity_ratio
    })

faults_df = pd.DataFrame(faults_list)

# Assign stable unique fault_id (F1, F2, ...)
faults_df = faults_df.sort_values(['severity_ratio'], ascending=False).reset_index(drop=True)
faults_df.insert(0, 'fault_id', ['F{}'.format(i+1) for i in range(len(faults_df))])

# Save faults_df in kernel variable (already assigned)

# 2) Filter out likely noise faults: drop length_in_minutes <=1 or severity_ratio <= 1.5
severity_cut = 1.5
faults_filtered_df = faults_df.loc[
    (faults_df['length_in_minutes'] > 1) & (faults_df['severity_ratio'] > severity_cut)
].copy()

# If no faults remain, lower threshold to 1.2 and report which threshold was used (store variable)
threshold_used_for_filter = severity_cut
if faults_filtered_df.empty:
    severity_cut = 1.2
    threshold_used_for_filter = severity_cut
    faults_filtered_df = faults_df.loc[
        (faults_df['length_in_minutes'] > 1) & (faults_df['severity_ratio'] > severity_cut)
    ].copy()

# 3) Prepare outputs
# a) Top 20 faults sorted by severity_ratio desc with requested columns
top20_faults = (
    faults_filtered_df
    .sort_values('severity_ratio', ascending=False)
    .loc[:, ['fault_id', 'cmdb_id', 'kpi_name', 'start_timestamp', 'end_timestamp',
             'length_in_minutes', 'max_value_in_interval', 'p95_threshold', 'severity_ratio']]
    .head(20)
    .reset_index(drop=True)
)

# b) Top 10 components by number of faults in faults_filtered_df
if faults_filtered_df.empty:
    top_fault_components_df = pd.DataFrame(columns=[
        'cmdb_id', 'fault_count', 'distinct_kpis_with_faults', 'earliest_fault_start'
    ])
else:
    top_fault_components_df = (
        faults_filtered_df
        .groupby('cmdb_id', dropna=False)
        .agg(
            fault_count=('fault_id', 'size'),
            distinct_kpis_with_faults=('kpi_name', 'nunique'),
            earliest_fault_start=('start_timestamp', 'min')
        )
        .reset_index()
        .sort_values('fault_count', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

# Keep faults_df and faults_filtered_df in kernel; return only the two compact outputs as required
top20_faults, top_fault_components_df

```
Out[5]:
```
```python
# Summarize the fault identification results in plain English as a compact string.
# Reuse kernel variables: faults_df, faults_filtered_df, top20_faults, top_fault_components_df, threshold_used_for_filter

try:
    faults_filtered_df
    top20_faults
    top_fault_components_df
    threshold_used_for_filter
except NameError:
    summary_text = "Fault variables not found in kernel."
else:
    total_faults = int(faults_filtered_df.shape[0])
    thresh = float(threshold_used_for_filter)
    # Top 5 faults summary
    top_faults_lines = []
    for _, r in top20_faults.head(5).iterrows():
        fid = r['fault_id']
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        start = r['start_timestamp']
        end = r['end_timestamp']
        length = int(r['length_in_minutes'])
        sev = float(r['severity_ratio']) if not pd.isna(r['severity_ratio']) else float('nan')
        top_faults_lines.append(f"{fid} | {cmdb} | {kpi} | {start} to {end} | len={length}m | sev={sev:.3f}")

    # Top components (top 5)
    top_comp_lines = []
    for _, r in top_fault_components_df.head(5).iterrows():
        cmdb = r['cmdb_id']
        cnt = int(r['fault_count'])
        distinct = int(r['distinct_kpis_with_faults'])
        first = r['earliest_fault_start']
        top_comp_lines.append(f"{cmdb}: faults={cnt}, distinct_kpis={distinct}, first_start={first}")

    summary_text = (
        f"Fault identification summary (incident window):\n"
        f"- Fault intervals detected after filtering: {total_faults}\n"
        f"- Filtering used severity_ratio threshold = {thresh} (length>1 minute also required).\n"
        f"- Sample top faults (up to 5):\n  " + "\n  ".join(top_faults_lines) + "\n"
        f"- Top components by fault count (up to 5):\n  " + "\n  ".join(top_comp_lines) + "\n\n"
        "Notes:\n"
        "- Filtering dropped single-minute spikes and low-severity intervals; threshold was lowered to include meaningful faults if needed.\n"
        "- Results are stored in kernel variables: faults_df (all intervals) and faults_filtered_df (filtered intervals)."
    )

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(  fault_id       cmdb_id                                           kpi_name            start_timestamp              end_timestamp  length_in_minutes  max_value_in_interval  p95_threshold  severity_ratio
0      F62  ServiceTest6                                            app.cnt  2021-03-23T15:03:00+08:00  2021-03-23T15:04:00+08:00                  2             550.000000     372.000000        1.478495
1      F67      Tomcat03                             OSLinux-CPU_CPU_CPUWio  2021-03-23T15:24:00+08:00  2021-03-23T15:25:00+08:00                  2               2.655015       1.879650        1.412505
2      F76          MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-23T15:00:00+08:00  2021-03-23T15:27:00+08:00                 28               0.400000       0.300000        1.333333
3      F88  ServiceTest2                                            app.mrt  2021-03-23T15:29:00+08:00  2021-03-23T15:30:00+08:00                  2               0.535540       0.422575        1.267325
4      F89  ServiceTest8                                            app.mrt  2021-03-23T15:29:00+08:00  2021-03-23T15:30:00+08:00                  2               0.607940       0.481213        1.263349
5      F90      Tomcat01                            OSLinux-CPU_CPU_CPULoad  2021-03-23T15:05:00+08:00  2021-03-23T15:06:00+08:00                  2               1.640000       1.300000        1.261538,         cmdb_id  fault_count  distinct_kpis_with_faults       earliest_fault_start
0          MG02            1                          1  2021-03-23T15:00:00+08:00
1  ServiceTest2            1                          1  2021-03-23T15:29:00+08:00
2  ServiceTest6            1                          1  2021-03-23T15:03:00+08:00
3  ServiceTest8            1                          1  2021-03-23T15:29:00+08:00
4      Tomcat01            1                          1  2021-03-23T15:05:00+08:00
5      Tomcat03            1                          1  2021-03-23T15:24:00+08:00)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables
try:
    faults_filtered_df
    trace_anomalies_agg_df
    log_incident_df
except NameError as e:
    raise

tz = pytz.timezone('Asia/Shanghai')

# 1) Top 5 components by earliest_fault_start (ascending)
if faults_filtered_df.empty:
    top5_faulty_components = pd.DataFrame(columns=['cmdb_id', 'earliest_fault_start', 'fault_count'])
else:
    # parse start timestamps to datetimes (aware)
    starts = pd.to_datetime(faults_filtered_df['start_timestamp'])
    faults_filtered_df['_start_dt'] = starts
    comp_summary = (
        faults_filtered_df
        .groupby('cmdb_id', dropna=False)
        .agg(earliest_fault_start=('_start_dt', 'min'),
             fault_count=('fault_id', 'size'))
        .reset_index()
    )
    comp_summary['earliest_fault_start'] = comp_summary['earliest_fault_start'].dt.tz_convert(tz).astype(str)
    top5_faulty_components = comp_summary.sort_values('earliest_fault_start', ascending=True).loc[:, ['cmdb_id', 'earliest_fault_start', 'fault_count']].head(5).reset_index(drop=True)

# 2) Scan trace_anomalies_agg_df for matches related to those top-5 components
# Build list of top5 cmdb ids
top5_list = top5_faulty_components['cmdb_id'].tolist()

if len(top5_list) == 0 or trace_anomalies_agg_df.empty:
    selected_trace_anomalies_for_top5 = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'anomaly_count',
                                                              'earliest_anomaly_timestamp', 'max_value', 'p95_threshold', 'severity_ratio'])
else:
    ta = trace_anomalies_agg_df.copy()
    # ensure columns exist and consistent names
    ta = ta.rename(columns={'max_value': 'max_value', 'max_value': 'max_value'})  # no-op to be safe

    # Condition A: cmdb_id in top5_list
    cond_a = ta['cmdb_id'].isin(top5_list)

    # Condition B: trace_name contains any of the cmdb ids as substring
    pattern = '|'.join([str(x) for x in top5_list])
    # Use na=False to avoid NaNs
    cond_b = ta['trace_name'].astype(str).str.contains(pattern, regex=True, na=False)

    matched = ta.loc[cond_a | cond_b].copy()

    # Select required columns and sort by anomaly_count desc, limit 50
    if not matched.empty:
        # Ensure column names in matched correspond to requested output
        cols_map = {
            'max_value': 'max_value',
            'p95_threshold': 'p95_threshold'
        }
        # Some upstream names might differ; align expected columns
        for c in ['anomaly_count', 'earliest_anomaly_timestamp', 'max_value', 'p95_threshold', 'severity_ratio']:
            if c not in matched.columns:
                matched[c] = np.nan

        selected_trace_anomalies_for_top5 = (
            matched
            .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value', 'p95_threshold', 'severity_ratio']]
            .sort_values('anomaly_count', ascending=False)
            .head(50)
            .reset_index(drop=True)
        )
    else:
        selected_trace_anomalies_for_top5 = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'anomaly_count',
                                                                  'earliest_anomaly_timestamp', 'max_value', 'p95_threshold', 'severity_ratio'])

# 3) Aggregate logs in incident window for top-5 components
# If log_incident_df not present or empty, report no logs for each component
if 'log_incident_df' not in globals() or log_incident_df.empty:
    # Create rows indicating no logs for each top5 component
    rows = []
    for cmdb in top5_list:
        rows.append({'cmdb_id': cmdb, 'log_name': '<NO_LOGS_IN_WINDOW>', 'count': 0})
    selected_logs_for_top5 = pd.DataFrame(rows, columns=['cmdb_id', 'log_name', 'count'])
else:
    logs = log_incident_df.copy()
    # Filter to top5 components
    logs_top5 = logs[logs['cmdb_id'].isin(top5_list)].copy()
    if logs_top5.empty:
        rows = []
        for cmdb in top5_list:
            rows.append({'cmdb_id': cmdb, 'log_name': '<NO_LOGS_IN_WINDOW>', 'count': 0})
        selected_logs_for_top5 = pd.DataFrame(rows, columns=['cmdb_id', 'log_name', 'count'])
    else:
        grouped_logs = (
            logs_top5
            .groupby(['cmdb_id', 'log_name'], dropna=False)
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False)
            .head(20)
            .reset_index(drop=True)
        )
        # Identify any top5 components missing from grouped_logs and add no-log rows
        present_cmdbs = set(grouped_logs['cmdb_id'].unique())
        missing = [c for c in top5_list if c not in present_cmdbs]
        if missing:
            missing_rows = pd.DataFrame([{'cmdb_id': c, 'log_name': '<NO_LOGS_IN_WINDOW>', 'count': 0} for c in missing])
            selected_logs_for_top5 = pd.concat([grouped_logs, missing_rows], ignore_index=True).reset_index(drop=True)
        else:
            selected_logs_for_top5 = grouped_logs

# Keep the specified kernel variables
top5_faulty_components = top5_faulty_components
selected_trace_anomalies_for_top5 = selected_trace_anomalies_for_top5
selected_logs_for_top5 = selected_logs_for_top5

# Return the three compact outputs only
top5_faulty_components, selected_trace_anomalies_for_top5, selected_logs_for_top5

```
Out[6]:
```
```python
# Summarize findings for top-5 faulty components, related trace anomalies, and logs (compact plain English).
# Reuse kernel variables: top5_faulty_components, selected_trace_anomalies_for_top5, selected_logs_for_top5

try:
    top5 = top5_faulty_components.copy()
    traces = selected_trace_anomalies_for_top5.copy()
    logs = selected_logs_for_top5.copy()
except NameError:
    summary = "Required kernel variables are missing."
else:
    # Top-5 components by earliest fault start
    top5_lines = []
    for _, r in top5.iterrows():
        top5_lines.append(f"{r['cmdb_id']} (first fault: {r['earliest_fault_start']}, faults: {int(r['fault_count'])})")
    top5_summary = "; ".join(top5_lines) if top5_lines else "No faulty components detected."

    # Trace anomalies summary: highlight top 3 trace groups by anomaly_count in the selected set
    trace_highlights = []
    if not traces.empty:
        for _, r in traces.sort_values('anomaly_count', ascending=False).head(3).iterrows():
            trace_highlights.append(
                f"{r['cmdb_id']} / {r['trace_name']}: anomalies={int(r['anomaly_count'])}, "
                f"max={float(r['max_value']):.3f}, thr={float(r['p95_threshold']):.3f}, sev={float(r['severity_ratio']):.3f}"
            )
    trace_summary = "\n  ".join(trace_highlights) if trace_highlights else "No related trace anomalies found for top-5 components."

    # Logs summary for top-5: show top log counts or note missing logs
    log_lines = []
    if logs.empty:
        log_summary = "No logs in the incident window for the top-5 components."
    else:
        # Show up to top 10 log rows
        for _, r in logs.head(10).iterrows():
            if r['log_name'] == '<NO_LOGS_IN_WINDOW>':
                log_lines.append(f"{r['cmdb_id']}: no logs in window")
            else:
                log_lines.append(f"{r['cmdb_id']} / {r['log_name']}: count={int(r['count'])}")
        log_summary = "; ".join(log_lines)

    summary = (
        "Compact result for top-5 faulty components and related telemetry:\n\n"
        f"1) Top-5 components by earliest fault start:\n  {top5_summary}\n\n"
        "2) Related trace anomalies (sample top 3):\n  " + trace_summary + "\n\n"
        "3) Logs in the incident window for these components (sample):\n  " + log_summary + "\n\n"
        "Notes:\n"
        "- Tomcat01 and Tomcat03 show strong trace.self.duration_p95 anomalies (highest anomaly counts).\n"
        "- IG01 emits traces referencing Tomcat01/03 (e.g., trace.to_Tomcat03.duration_mean) with notable severity in a few groups.\n"
        "- Tomcat01 and Tomcat03 have abundant log activity (log.error_count and log.row_count = 31); MG02 and the ServiceTest components report no logs in the window.\n"
        "- All intermediate result tables are stored in kernel variables for further localization steps."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id       earliest_fault_start  fault_count
0          MG02  2021-03-23 15:00:00+08:00            1
1  ServiceTest6  2021-03-23 15:03:00+08:00            1
2      Tomcat01  2021-03-23 15:05:00+08:00            1
3      Tomcat03  2021-03-23 15:24:00+08:00            1
4  ServiceTest2  2021-03-23 15:29:00+08:00            1,      cmdb_id                       trace_name  anomaly_count earliest_anomaly_timestamp   max_value  p95_threshold  severity_ratio
0   Tomcat01          trace.self.duration_p95             17        2021-03-23T07:01:00    1.705450       1.004000        1.698655
1   Tomcat03          trace.self.duration_p95             16        2021-03-23T07:01:00    1.500900       1.004000        1.494920
2       IG01      trace.to_Tomcat01.row_count              2        2021-03-23T07:23:00  325.000000     311.000000        1.045016
3       IG01  trace.to_Tomcat03.duration_mean              2        2021-03-23T07:27:00    2.923375       1.046730        2.792864
4       IG01  trace.to_Tomcat01.duration_mean              2        2021-03-23T07:21:00    1.231567       0.969692        1.270060
..       ...                              ...            ...                        ...         ...            ...             ...
45      MG02          trace.self.duration_p95              2        2021-03-23T07:27:00    5.283500       2.725300        1.938686
46      MG02  trace.to_dockerB1.duration_mean              2        2021-03-23T07:27:00    1.448200       1.110353        1.304270
47      MG02   trace.to_dockerB1.duration_p95              2        2021-03-23T07:21:00    6.577050       4.891950        1.344464
48      MG02      trace.to_dockerB1.row_count              2        2021-03-23T07:02:00  296.000000     281.000000        1.053381
49      MG02   trace.to_dockerA1.duration_p95              2        2021-03-23T07:20:00    2.073850       2.026750        1.023239

[50 rows x 7 columns],         cmdb_id             log_name  count
0      Tomcat01      log.error_count     31
1      Tomcat01        log.row_count     31
2      Tomcat03        log.row_count     31
3      Tomcat03      log.error_count     31
4      Tomcat03     log.gc_pause_max     25
5      Tomcat01     log.gc_pause_max     24
6          MG02  <NO_LOGS_IN_WINDOW>      0
7  ServiceTest6  <NO_LOGS_IN_WINDOW>      0
8  ServiceTest2  <NO_LOGS_IN_WINDOW>      0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables
try:
    trace_anomalies_agg_df
except NameError:
    trace_anomalies_agg_df = pd.DataFrame()

try:
    log_incident_df
except NameError:
    log_incident_df = pd.DataFrame()

try:
    error_logs_df
except NameError:
    error_logs_df = pd.DataFrame()

# 1) Select traces for localization
substrs = [
    'to_MG02', 'from_MG02', 'to_MG01', 'from_MG01',
    'to_Mysql', 'from_Mysql', 'to_Tomcat', 'from_Tomcat'
]
cmdb_targets = {'MG02','Tomcat01','Tomcat03','Tomcat04','Tomcat02','Mysql01','Mysql02'}

if trace_anomalies_agg_df.empty:
    selected_traces_for_localization = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp',
        'max_value', 'p95_threshold', 'severity_ratio'
    ])
else:
    ta = trace_anomalies_agg_df.copy()
    # Build regex pattern to match any substring (escape to be safe)
    pattern = "(" + "|".join(re.escape(s) for s in substrs) + ")"
    cond_a = ta['cmdb_id'].isin(cmdb_targets)
    cond_b = ta['trace_name'].astype(str).str.contains(pattern, regex=True, na=False)
    matched = ta.loc[cond_a | cond_b].copy()
    # Ensure required columns exist
    for c in ['anomaly_count','earliest_anomaly_timestamp','max_value','p95_threshold','severity_ratio']:
        if c not in matched.columns:
            matched[c] = np.nan
    selected_traces_for_localization = (
        matched
        .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value', 'p95_threshold', 'severity_ratio']]
        .sort_values('severity_ratio', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )

# 2) Extract Tomcat01 and Tomcat03 logs from log_incident_df (incident window)
tomcats = ['Tomcat01','Tomcat03']
if log_incident_df.empty:
    tomcat_logs_sample = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
else:
    logs = log_incident_df.copy()
    # Convert timestamp to local ISO for readability
    logs['_ts_local'] = pd.to_datetime(logs['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    tomcat_logs_sample = (
        logs.loc[logs['cmdb_id'].isin(tomcats), ['_ts_local','cmdb_id','log_name','value']]
        .rename(columns={'_ts_local':'timestamp'})
        .sort_values('timestamp', ascending=True)
        .head(50)
        .reset_index(drop=True)
    )

# 3) Extract error logs mentioning top candidate components
candidates = ['MG02','Tomcat01','Tomcat03','Mysql01','Mysql02']
if error_logs_df.empty:
    error_logs_sample = pd.DataFrame([{'indicator':'NO_MATCHING_ERROR_LOGS'}])
else:
    el = error_logs_df.copy()
    # Convert timestamp to local ISO if timestamp exists
    if 'timestamp' in el.columns and not el['timestamp'].isnull().all():
        el['_ts_local'] = pd.to_datetime(el['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        el['_ts_local'] = pd.NaT
    # Match by cmdb_id or substring in message
    cond_cmdb = el['cmdb_id'].isin(candidates)
    # message may be missing; ensure string
    el['message_str'] = el.get('message', '').astype(str)
    pattern_msgs = "(" + "|".join(re.escape(x) for x in candidates) + ")"
    cond_msg = el['message_str'].str.contains(pattern_msgs, regex=True, na=False)
    matched_err = el.loc[cond_cmdb | cond_msg].copy()
    if matched_err.empty:
        error_logs_sample = pd.DataFrame([{'indicator':'NO_MATCHING_ERROR_LOGS'}])
    else:
        # select up to 20 rows, convert timestamp to ISO string for compactness
        matched_err['timestamp_local'] = matched_err['_ts_local'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
        error_logs_sample = matched_err.loc[:, ['timestamp_local','cmdb_id','message_str']].head(20).rename(columns={'timestamp_local':'timestamp','message_str':'message'}).reset_index(drop=True)

# Save results in kernel variables (already assigned) and return compact outputs
selected_traces_for_localization, tomcat_logs_sample, error_logs_sample

```
Out[7]:
```
```python
# Build a compact plain-English summary from kernel variables about targeted trace & log inspection.
# Reuse: top5_faulty_components, selected_traces_for_localization, tomcat_logs_sample, error_logs_sample

import math

try:
    top5 = top5_faulty_components.copy()
    traces = selected_traces_for_localization.copy()
    tomcat_logs = tomcat_logs_sample.copy()
    err_logs = error_logs_sample.copy()
except NameError:
    summary_text = "Required kernel variables are missing."
else:
    # 1) Top-5 components (earliest fault start)
    top5_lines = []
    for _, r in top5.iterrows():
        top5_lines.append(f"{r['cmdb_id']} (first fault: {r['earliest_fault_start']}, faults: {int(r['fault_count'])})")
    top5_summary = "; ".join(top5_lines) if top5_lines else "No faulty components found."

    # 2) Top trace anomalies from selected_traces_for_localization (top 3 by severity_ratio)
    trace_lines = []
    if traces.empty:
        trace_summary = "No selected trace anomalies matched the criteria."
    else:
        top_traces = traces.sort_values('severity_ratio', ascending=False).head(3)
        for _, r in top_traces.iterrows():
            cmdb = r['cmdb_id']
            name = r['trace_name']
            cnt = int(r['anomaly_count']) if not pd.isna(r['anomaly_count']) else 0
            mv = float(r['max_value']) if not pd.isna(r['max_value']) else float('nan')
            thr = float(r['p95_threshold']) if not pd.isna(r['p95_threshold']) else float('nan')
            sev = float(r['severity_ratio']) if not pd.isna(r['severity_ratio']) else float('nan')
            mv_s = f"{mv:.3f}" if not math.isnan(mv) else "NaN"
            thr_s = f"{thr:.3f}" if not math.isnan(thr) else "NaN"
            sev_s = f"{sev:.3f}" if not math.isnan(sev) else "NaN"
            trace_lines.append(f"{cmdb} / {name}: anomalies={cnt}, max={mv_s}, thr={thr_s}, sev={sev_s}")
        trace_summary = "; ".join(trace_lines)

    # 3) Tomcat logs sample summary
    if tomcat_logs.empty:
        logs_summary = "No Tomcat logs in the incident window for Tomcat01/Tomcat03."
    else:
        # counts per cmdb_id in sample
        log_counts = tomcat_logs['cmdb_id'].value_counts().to_dict()
        log_count_lines = [f"{k}: {v} rows" for k, v in log_counts.items()]
        # show earliest sample timestamp
        first_ts = tomcat_logs['timestamp'].min() if 'timestamp' in tomcat_logs.columns and not tomcat_logs['timestamp'].isnull().all() else None
        logs_summary = f"Sampled Tomcat logs counts: {', '.join(log_count_lines)}; earliest_sample={first_ts}"

    # 4) Error logs presence
    if ('indicator' in err_logs.columns and err_logs['indicator'].iloc[0] == 'NO_MATCHING_ERROR_LOGS') or err_logs.empty:
        err_summary = "No matching error_logs for top components in the incident window."
    else:
        err_count = err_logs.shape[0]
        err_summary = f"{err_count} matching error_log rows (sample returned)."

    # Compose final compact summary
    summary_text = (
        "Targeted localization findings (compact):\n\n"
        f"1) Top-5 faulty components by earliest fault:\n  {top5_summary}\n\n"
        f"2) Top related trace anomalies (sample top 3 by severity):\n  {trace_summary}\n\n"
        f"3) Tomcat logs (Tomcat01 & Tomcat03) sample:\n  {logs_summary}\n\n"
        f"4) Error logs:\n  {err_summary}\n\n"
        "Notes:\n"
        "- High-severity traces include cross-service durations (e.g., traces involving MG01/MG02 and Tomcat nodes).\n"
        "- Tomcat01/Tomcat03 have abundant log rows in the incident window (see sample); no matching error log messages were found for the primary candidate components."
    )

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  anomaly_count earliest_anomaly_timestamp  max_value  p95_threshold  severity_ratio
0       MG01  trace.from_Tomcat04.duration_mean              2        2021-03-23T07:27:00   4.836000       0.853033        5.669183
1   Tomcat04        trace.to_MG01.duration_mean              2        2021-03-23T07:27:00   4.836000       0.853033        5.669183
2   dockerB1      trace.from_MG01.duration_mean              2        2021-03-23T07:21:00   5.783667       1.088315        5.314334
3   dockerB2      trace.from_MG01.duration_mean              2        2021-03-23T07:27:00   7.819000       1.657630        4.716975
4   Tomcat01      trace.from_IG02.duration_mean              2        2021-03-23T07:27:00   4.207333       1.002915        4.195104
..       ...                                ...            ...                        ...        ...            ...             ...
45      MG02   trace.from_dockerB2.duration_p95              2        2021-03-23T07:26:00   3.316500       1.832100        1.810218
46      IG01     trace.to_Tomcat04.duration_p95              2        2021-03-23T07:27:00   8.706400       4.834500        1.800889
47  Tomcat04       trace.from_IG01.duration_p95              2        2021-03-23T07:27:00   8.706400       4.834500        1.800889
48  dockerB2        trace.to_MG01.duration_mean              2        2021-03-23T07:27:00   0.845000       0.475801        1.775953
49  dockerA2        trace.to_MG02.duration_mean              2        2021-03-23T07:27:00   1.072000       0.611897        1.751927

[50 rows x 7 columns],                    timestamp   cmdb_id          log_name   value
0  2021-03-23 15:00:00+08:00  Tomcat01   log.error_count    0.00
1  2021-03-23 15:00:00+08:00  Tomcat01  log.gc_pause_max    0.07
2  2021-03-23 15:00:00+08:00  Tomcat01     log.row_count  487.00
3  2021-03-23 15:00:00+08:00  Tomcat03   log.error_count    0.00
4  2021-03-23 15:00:00+08:00  Tomcat03  log.gc_pause_max    0.02
..                       ...       ...               ...     ...
45 2021-03-23 15:08:00+08:00  Tomcat01   log.error_count    0.00
46 2021-03-23 15:08:00+08:00  Tomcat01  log.gc_pause_max    0.08
47 2021-03-23 15:08:00+08:00  Tomcat01     log.row_count  486.00
48 2021-03-23 15:08:00+08:00  Tomcat03   log.error_count    0.00
49 2021-03-23 15:08:00+08:00  Tomcat03  log.gc_pause_max    0.09

[50 rows x 4 columns],                 indicator
0  NO_MATCHING_ERROR_LOGS)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: faults_filtered_df, anomalies_agg_df, trace_anomalies_agg_df, log_incident_df
# Assume they exist from prior steps.

# 1) mg02_faults from faults_filtered_df
if 'faults_filtered_df' in globals() and not faults_filtered_df.empty:
    # Select and align required columns; some names may vary, handle missing gracefully
    ff = faults_filtered_df.copy()
    cols_map = {
        'fault_id': 'fault_id',
        'cmdb_id': 'cmdb_id',
        'kpi_name': 'kpi_name',
        'start_timestamp': 'start_timestamp',
        'end_timestamp': 'end_timestamp',
        'length_in_minutes': 'length_in_minutes',
        'max_value_in_interval': 'max_value_in_interval',
        'p95_threshold': 'p95_threshold',
        'severity_ratio': 'severity_ratio'
    }
    for k in cols_map:
        if cols_map[k] not in ff.columns:
            ff[cols_map[k]] = np.nan
    mg02_faults = ff.loc[ff['cmdb_id'] == 'MG02', list(cols_map.values())].reset_index(drop=True)
else:
    mg02_faults = pd.DataFrame(columns=['fault_id','cmdb_id','kpi_name','start_timestamp','end_timestamp',
                                        'length_in_minutes','max_value_in_interval','p95_threshold','severity_ratio'])

# 2) mg02_metric_anoms from anomalies_agg_df (top 10 by anomaly_count)
if 'anomalies_agg_df' in globals() and not anomalies_agg_df.empty:
    ma = anomalies_agg_df.copy()
    # Ensure expected columns exist, create if missing
    for c in ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_threshold','severity_ratio']:
        if c not in ma.columns:
            ma[c] = np.nan
    mg02_metric_anoms = (
        ma.loc[ma['cmdb_id'] == 'MG02', 
               ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_threshold','severity_ratio']]
          .sort_values('anomaly_count', ascending=False)
          .head(10)
          .reset_index(drop=True)
    )
else:
    mg02_metric_anoms = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_threshold','severity_ratio'])

# 3) mg02_trace_anoms from trace_anomalies_agg_df (filter by cmdb_id or trace_name contains MG02)
substrs = ['to_MG02','from_MG02']
if 'trace_anomalies_agg_df' in globals() and not trace_anomalies_agg_df.empty:
    ta = trace_anomalies_agg_df.copy()
    # Ensure expected columns
    for c in ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_threshold','severity_ratio']:
        if c not in ta.columns:
            ta[c] = np.nan
    cond = (ta['cmdb_id'] == 'MG02') | (ta['trace_name'].astype(str).str.contains('|'.join(substrs), na=False))
    mg02_trace_anoms = (
        ta.loc[cond, ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_threshold','severity_ratio']]
          .sort_values('severity_ratio', ascending=False)
          .head(10)
          .reset_index(drop=True)
    )
else:
    mg02_trace_anoms = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','max_value','p95_threshold','severity_ratio'])

# 4) mg02_logs_sample from log_incident_df (up to 5 rows) or indicator if none
if 'log_incident_df' in globals() and not log_incident_df.empty:
    logs = log_incident_df.copy()
    # Filter MG02 rows
    mg_logs = logs.loc[logs['cmdb_id'] == 'MG02'].copy()
    if not mg_logs.empty:
        # Convert timestamp to local ISO for readability
        mg_logs['_ts_local'] = pd.to_datetime(mg_logs['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        mg02_logs_sample = (
            mg_logs.loc[:, ['_ts_local','cmdb_id','log_name','value']]
            .rename(columns={'_ts_local':'timestamp'})
            .sort_values('timestamp', ascending=True)
            .head(5)
            .reset_index(drop=True)
        )
    else:
        mg02_logs_sample = pd.DataFrame([{'indicator':'NO_LOGS_IN_WINDOW'}])
else:
    mg02_logs_sample = pd.DataFrame([{'indicator':'NO_LOGS_IN_WINDOW'}])

# Keep these variables in kernel for next steps.
mg02_faults, mg02_metric_anoms, mg02_trace_anoms, mg02_logs_sample

```
Out[8]:
```
```python
# Build a compact MG02-focused diagnostic summary using existing kernel variables:
# mg02_faults, mg02_metric_anoms, mg02_trace_anoms, mg02_logs_sample.
# Produce a short plain-English summary string and display it.

import pandas as pd
import math

try:
    mg02_faults
    mg02_metric_anoms
    mg02_trace_anoms
    mg02_logs_sample
except NameError:
    summary = "MG02 diagnostic variables are missing from the kernel."
else:
    lines = []
    # 1) Fault intervals
    if isinstance(mg02_faults, pd.DataFrame) and not mg02_faults.empty:
        for _, r in mg02_faults.iterrows():
            lines.append(
                f"Fault {r.get('fault_id')} | {r.get('kpi_name')} | {r.get('start_timestamp')} to {r.get('end_timestamp')} "
                f"len={int(r.get('length_in_minutes',0))}m | max={float(r.get('max_value_in_interval')):.3f} | "
                f"p95={float(r.get('p95_threshold')) if pd.notna(r.get('p95_threshold')) else 'NaN'} | sev={float(r.get('severity_ratio')):.3f}"
            )
    else:
        lines.append("No filtered faults for MG02.")

    # 2) Metric anomaly summary (top rows)
    if isinstance(mg02_metric_anoms, pd.DataFrame) and not mg02_metric_anoms.empty:
        metric_top = mg02_metric_anoms.head(5)
        metric_lines = []
        for _, r in metric_top.iterrows():
            metric_lines.append(
                f"{r['kpi_name']} (count={int(r['anomaly_count'])}) max={float(r['max_value']):.3e} thr={float(r['p95_threshold']):.3e} sev={float(r['severity_ratio']):.3f}"
            )
        metric_summary = " | ".join(metric_lines)
    else:
        metric_summary = "No metric anomalies for MG02."

    # 3) Trace anomaly summary (top rows)
    if isinstance(mg02_trace_anoms, pd.DataFrame) and not mg02_trace_anoms.empty:
        trace_top = mg02_trace_anoms.head(5)
        trace_lines = []
        for _, r in trace_top.iterrows():
            trace_lines.append(
                f"{r['cmdb_id']} / {r['trace_name']} (count={int(r['anomaly_count'])}) max={float(r['max_value']):.3f} thr={float(r['p95_threshold']):.3f} sev={float(r['severity_ratio']):.3f}"
            )
        trace_summary = " | ".join(trace_lines)
    else:
        trace_summary = "No trace anomalies related to MG02."

    # 4) Logs presence
    if isinstance(mg02_logs_sample, pd.DataFrame) and not mg02_logs_sample.empty and 'indicator' not in mg02_logs_sample.columns:
        # summarize up to 5 rows present
        count_logs = mg02_logs_sample.shape[0]
        first_ts = mg02_logs_sample['timestamp'].min()
        logs_summary = f"{count_logs} log rows in window (sample earliest={first_ts})"
    else:
        logs_summary = "NO_LOGS_IN_WINDOW for MG02."

    # Compose final compact summary
    summary = (
        "MG02 diagnostic summary (compact):\n\n"
        "Fault intervals (filtered):\n  " + ("\n  ".join(lines)) + "\n\n"
        "Metric anomalies (top sample):\n  " + metric_summary + "\n\n"
        "Trace anomalies (top sample):\n  " + trace_summary + "\n\n"
        "Logs:\n  " + logs_summary + "\n\n"
        "Key observations:\n"
        "- There is one multi-minute fault interval involving a zabbix agentd process on MG02 (long: 28m, sev~1.33).\n"
        "- Metric anomalies on MG02 include JVM memory max (constant) and process-level metrics; many metric severities are near 1.0 (not large).\n"
        "- Stronger signals appear in traces: multiple high-severity cross-service traces involve MG02 (e.g., to/from dockerA2, Tomcat nodes) with severity ratios >2 — this suggests latency or slow downstream behavior involving MG02.\n"
        "- No application logs were captured for MG02 in the incident window (NO_LOGS_IN_WINDOW), limiting log-based diagnosis.\n\n"
        "Stored kernel variables for follow-up: mg02_faults, mg02_metric_anoms, mg02_trace_anoms, mg02_logs_sample."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  fault_id cmdb_id                                           kpi_name            start_timestamp              end_timestamp  length_in_minutes  max_value_in_interval  p95_threshold  severity_ratio
0      F76    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-23T15:00:00+08:00  2021-03-23T15:27:00+08:00                 28                    0.4            0.3        1.333333,   cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp     max_value  p95_threshold  severity_ratio
0    MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31        2021-03-23T07:00:00  2.040136e+09   2.040136e+09        1.000000
1    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             29        2021-03-23T07:00:00  4.000000e-01   3.000000e-01        1.333333
2    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             29        2021-03-23T07:00:00  1.000000e+00   1.000000e+00        1.000000
3    MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             29        2021-03-23T07:00:00  3.000000e+00   3.000000e+00        1.000000
4    MG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc             20        2021-03-23T07:00:00  9.800000e+01   9.800000e+01        1.000000
5    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc             16        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
6    MG02    OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent             16        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
7    MG02                       OSLinux-OSLinux_SWAP_SWAP_Si             16        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
8    MG02                       OSLinux-OSLinux_SWAP_SWAP_So             16        2021-03-23T07:00:00  0.000000e+00   0.000000e+00             NaN
9    MG02                    OSLinux-CPU_CPU-3_SingleCpuidle             15        2021-03-23T07:01:00  0.000000e+00   0.000000e+00             NaN,     cmdb_id                         trace_name  anomaly_count earliest_anomaly_timestamp  max_value  p95_threshold  severity_ratio
0  dockerA2         trace.to_MG02.duration_p95              2        2021-03-23T07:26:00   4.775650       1.728900        2.762248
1      MG02   trace.from_dockerA2.duration_p95              2        2021-03-23T07:26:00   4.775650       1.728900        2.762248
2      MG02    trace.to_dockerA2.duration_mean              2        2021-03-23T07:29:00   0.965625       0.376787        2.562787
3  dockerA2      trace.from_MG02.duration_mean              2        2021-03-23T07:29:00   0.965625       0.376787        2.562787
4      MG02  trace.from_Tomcat01.duration_mean              2        2021-03-23T07:27:00   1.461750       0.657312        2.223830
5  Tomcat01        trace.to_MG02.duration_mean              2        2021-03-23T07:27:00   1.461750       0.657312        2.223830
6  dockerB1         trace.to_MG02.duration_p95              2        2021-03-23T07:24:00   3.210000       1.589300        2.019757
7      MG02   trace.from_dockerB1.duration_p95              2        2021-03-23T07:24:00   3.210000       1.589300        2.019757
8  Tomcat02        trace.to_MG02.duration_mean              2        2021-03-23T07:27:00   1.109167       0.564456        1.965019
9      MG02  trace.from_Tomcat02.duration_mean              2        2021-03-23T07:27:00   1.109167       0.564456        1.965019,            indicator
0  NO_LOGS_IN_WINDOW)```
```